# 05 – Đánh giá & so sánh (Evaluation)
Bảng tổng hợp toàn bộ thí nghiệm trong `spam_filter/experiments.py`, metric đầy đủ, confusion matrix, ROC/PR.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

In [ ]:
from spam_filter.experiments import EXPERIMENTS
from spam_filter.runner import run_specs

results, pipes = run_specs(EXPERIMENTS, X_train, y_train, X_test, y_test, cfg.cv_folds)
results.drop(columns="params").round(4)

## 1. Tham số từng thí nghiệm (để tái lập)

In [ ]:
results[["id", "params"]]

## 2. So sánh Precision / Recall / F1 (lớp spam)

In [ ]:
results.set_index("id")[["precision", "recall", "f1"]].plot.bar(rot=0, figsize=(9, 4), ylim=(0.7, 1.0), title="Test – lớp spam");

## 3. Confusion matrix: baseline và mô hình tốt nhất (chọn theo CV)

In [ ]:
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
best_id = results.sort_values("cv_f1_train", ascending=False).iloc[0]["id"]
print("Mô hình tốt nhất theo CV:", best_id)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, i in zip(axes, ["E0", best_id]):
    ConfusionMatrixDisplay.from_predictions(y_test, pipes[i].predict(X_test), display_labels=["ham", "spam"], cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(i)
plt.tight_layout()
print(classification_report(y_test, pipes[best_id].predict(X_test), target_names=["ham", "spam"], digits=4))

## 4. ROC và Precision-Recall

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i in ["E0", "E3", best_id]:
    RocCurveDisplay.from_estimator(pipes[i], X_test, y_test, name=i, ax=axes[0])
    PrecisionRecallDisplay.from_estimator(pipes[i], X_test, y_test, name=i, ax=axes[1])
axes[0].set_title("ROC"); axes[1].set_title("Precision-Recall (lớp spam)")
plt.tight_layout()

## Nhận xét (nhóm điền)
So sánh các phương án có căn cứ; vì sao không chỉ nhìn Accuracy khi lớp mất cân bằng? Chênh lệch nào có ý nghĩa khi test có ít mẫu spam, và đâu là biến động theo split?